# Demo de punta a punta (Fase 10)

Corre los 6 casos obligatorios del guion (`docs/guion-demo.md`) más 3 casos adicionales, invocando el orquestador real (Gemini + Postgres, checkpointer real) directamente en Python -- plan B si el túnel de WhatsApp no está disponible el día de la defensa (ver `docs/setup-whatsapp.md`).

Requiere: `docker compose up -d db` corriendo y al menos una `GEMINI_API_KEY_*` real en `.env`.

In [ ]:
import os
import sys
import uuid
from pathlib import Path

# Jupyter fija el cwd en la carpeta de la notebook; nos paramos en la raíz
# del proyecto para que config.py encuentre .env con una ruta relativa.
os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd() / "src"))

import psycopg

from fitosanitarios.config import get_settings
from fitosanitarios.orquestador.agente import (
    checkpointer_postgres,
    crear_agente,
    crear_modelo_chat_gemini,
)
from fitosanitarios.orquestador.estado import ContadorRepreguntas
from fitosanitarios.orquestador.turno import ejecutar_turno

settings = get_settings()
assert settings.gemini_api_keys, "Falta al menos una GEMINI_API_KEY_* en .env para esta demo"
print(f"Modelo: {settings.gemini_model} -- {len(settings.gemini_api_keys)} key(s) configurada(s)")

# Id único por corrida: el checkpointer es Postgres real y persiste entre
# corridas de esta notebook. Con un thread_id fijo, repetir un ensayo antes
# de la defensa hace que el LLM "recuerde" la conversación anterior y a
# veces no vuelva a llamar la tool (ver DIFICULTADES.md, Fase 10) -- por
# eso cada corrida arranca threads nuevos.
SESION = uuid.uuid4().hex[:8]
print(f"Sesión de esta corrida: {SESION}")

In [ ]:
model = crear_modelo_chat_gemini(settings)
pila_checkpointer = checkpointer_postgres(settings.database_url)
checkpointer = pila_checkpointer.__enter__()
agente = crear_agente(model, checkpointer=checkpointer)
contador = ContadorRepreguntas()
conn_log = psycopg.connect(settings.database_url)
print("Agente listo.")

In [ ]:
def turno(thread_id: str, texto: str) -> None:
    print(f"\n\U0001f9d1\u200d\U0001f33e ({thread_id}): {texto}")
    _, mensajes = ejecutar_turno(agente, thread_id, texto, contador, conn_log=conn_log)
    for m in mensajes:
        print(f"\U0001f916: {m}\n")

## Caso 1 -- Dictamen APTA

In [ ]:
turno(
    f"demo-e2e-{SESION}-1",
    "Quiero validar la receta completa: Flyer 10 Ec en soja, 170 cm3/ha, terrestre, "
    "en El Trébol, contra chinche de la alfalfa",
)

## Caso 2 -- Dictamen OBSERVADA (distancia insuficiente + dosis fuera de rango)

In [ ]:
turno(
    f"demo-e2e-{SESION}-2",
    "Quiero validar la receta completa: Flyer 10 Ec en soja, 500 cm3/ha, terrestre, "
    "lote -32.92877865019107,-60.6505, contra chinche de la alfalfa",
)

## Caso 3 -- Consulta de productos (listado)

In [ ]:
turno(f"demo-e2e-{SESION}-3", "¿Qué productos hay registrados para yuyo colorado en soja?")

## Caso 4 -- Repregunta agrupada

In [ ]:
turno(f"demo-e2e-{SESION}-4", "Quiero evaluar si puedo aplicar en mi lote")

## Caso 5 -- Fuera de dominio

In [ ]:
turno(f"demo-e2e-{SESION}-5", "¿Va a llover mañana en Rosario?")

## Caso 6 -- No resuelto (jurisdicci\u00f3n no cubierta)

In [ ]:
turno(
    f"demo-e2e-{SESION}-6",
    "Quiero evaluar riesgo para un lote en -34.6, -58.4 (Buenos Aires), Flyer 10 Ec, "
    "soja, terrestre, 2 L/ha, contra chinche de la alfalfa",
)

## Caso 7 (adicional) -- Consulta normativa con cita verificada

In [ ]:
turno(
    f"demo-e2e-{SESION}-7",
    "¿A qué distancia de una escuela puedo aplicar por tierra en Sastre?",
)

## Caso 8 (adicional) -- Extensiones Fase 9: registrar evento + agenda

In [ ]:
turno(f"demo-e2e-{SESION}-8", "Empecé a aplicar con la mosquito en el lote 4")
turno(f"demo-e2e-{SESION}-8", "Terminé de aplicar")
turno(f"demo-e2e-{SESION}-8", "¿qué tengo para hoy?")

## Caso 9 (adicional) -- Ambig\u00fcedad de producto

In [ ]:
turno(f"demo-e2e-{SESION}-9", "¿El glifosato está habilitado para soja?")

## Limpieza

In [ ]:
conn_log.close()
pila_checkpointer.__exit__(None, None, None)
print("Conexiones cerradas.")